In [1]:
import subprocess, sys

packages = [
    "groq",
    "langchain",
    "langchain-community",
    "faiss-cpu",
    "sentence-transformers",
    "PyMuPDF",
    "langchain-huggingface",
    "pandas"
]

for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

print("Packages installed successfully.")

Packages installed successfully.


In [2]:
from google.colab import files

uploaded = files.upload()
pdf_paths = list(uploaded.keys())

print("Uploaded PDFs:")
for p in pdf_paths:
    print(p)

Saving Brandenburg v. Ohio (1969.pdf to Brandenburg v. Ohio (1969.pdf
Saving matal_v._tam.pdf to matal_v._tam.pdf
Saving New York Times v. Sullivan (1964.pdf to New York Times v. Sullivan (1964.pdf
Saving Snyder v. Phelps (2011.pdf to Snyder v. Phelps (2011.pdf
Saving Tinker v. Des Monies.pdf to Tinker v. Des Monies.pdf
Uploaded PDFs:
Brandenburg v. Ohio (1969.pdf
matal_v._tam.pdf
New York Times v. Sullivan (1964.pdf
Snyder v. Phelps (2011.pdf
Tinker v. Des Monies.pdf


In [3]:
# Load PDFs and split into chunks
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []

for path in pdf_paths:
    docs.extend(PyMuPDFLoader(path).load())

chunks = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ".", " "]
).split_documents(docs)

print(f"Total documents loaded: {len(docs)}")
print(f"Total chunks created: {len(chunks)}")

Total documents loaded: 164
Total chunks created: 779


In [4]:
# Create FAISS vector store
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embedding_model)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("FAISS vector store created successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS vector store created successfully.


In [10]:
# Groq setup
from groq import Groq
import time
import pandas as pd
import re

GROQ_KEY = "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"

client = Groq(api_key=GROQ_KEY)

MODELS = {
    "Llama-3.1-8B": "llama-3.1-8b-instant",
    "Llama-3.3-70B": "llama-3.3-70b-versatile",
    "Llama-4-Scout": "meta-llama/llama-4-scout-17b-16e-instruct"
}

print("Groq client ready.")

Groq client ready.


In [11]:
# Helper function to query LLM
def query_llm(model_id, prompt):
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=400,
        temperature=0.1
    )
    return response.choices[0].message.content.strip()

In [12]:
# Questions
questions = [
    "What was the Supreme Court's ruling in Tinker v. Des Moines?",
    "What legal standard did the court establish in Brandenburg v. Ohio?",
    "What is the 'actual malice' standard from New York Times v. Sullivan?",
    "Who wrote the majority opinion in Snyder v. Phelps and what was the vote?",
    "Why did the court rule in favor of the Westboro Baptist Church in Snyder v. Phelps?",
    "How did the court balance free speech against potential harm in Brandenburg v. Ohio?",
    "Why did the court protect the students' speech in Tinker v. Des Moines?",
    "Is there a consistent standard across these cases for when speech can be restricted?",
    "How might the Tinker ruling apply to student social media posts today?",
    "How do these rulings collectively define the limits of First Amendment protection?"
]

ground_truth_answers = {
    1: "The Supreme Court ruled that students do not lose their First Amendment rights at school. Schools can restrict student speech only when it materially and substantially disrupts school operations.",
    2: "The Court established the imminent lawless action standard. Speech can be restricted only if it is directed to inciting imminent lawless action and is likely to produce such action.",
    3: "Actual malice means a public official must prove that a defamatory statement was made with knowledge that it was false or with reckless disregard for whether it was false.",
    4: "Chief Justice John Roberts wrote the majority opinion in Snyder v. Phelps. The vote was 8-1.",
    5: "The Court ruled for Westboro Baptist Church because the speech addressed matters of public concern and occurred in a public place, so it received strong First Amendment protection.",
    6: "The Court protected advocacy unless it was intended and likely to produce imminent lawless action.",
    7: "The Court protected the students because wearing armbands was symbolic speech and did not materially or substantially disrupt school activities.",
    8: "Across the cases, speech can usually be restricted only when there is a strong legal justification such as material disruption, actual malice, or imminent lawless action.",
    9: "Tinker may apply to student social media posts if the speech causes or is reasonably expected to cause a material and substantial disruption at school.",
    10: "Together, these cases show that the First Amendment strongly protects political, public, symbolic, and controversial speech, but does not protect speech that meets narrow exceptions such as defamation with actual malice or incitement of imminent lawless action."
}

In [13]:
# Build prompt
def build_prompt(question, retrieved_chunks):
    context = "\n\n---\n\n".join([c.page_content for c in retrieved_chunks])

    return f"""
You are a legal assistant. Answer using ONLY the context below.

Rules:
1. Do not use outside knowledge.
2. Do not fabricate case names, citations, quotes, or facts.
3. If the answer is not in the context, say: "I cannot find this in the documents."
4. Keep the answer clear and concise.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

In [14]:
# Run RAG system and save results
results = []

for i, question in enumerate(questions):
    print(f"Running question {i+1}/10...")

    retrieved = retriever.invoke(question)
    prompt = build_prompt(question, retrieved)

    row = {
        "Q#": i + 1,
        "Question": question
    }

    for model_name, model_id in MODELS.items():
        print(f"  Running model: {model_name}")
        row[model_name] = query_llm(model_id, prompt)
        time.sleep(1)

    results.append(row)

df = pd.DataFrame(results)
df.to_csv("lexrag_results.csv", index=False)

print("RAG results saved as lexrag_results.csv")

Running question 1/10...
  Running model: Llama-3.1-8B
  Running model: Llama-3.3-70B
  Running model: Llama-4-Scout
Running question 2/10...
  Running model: Llama-3.1-8B
  Running model: Llama-3.3-70B
  Running model: Llama-4-Scout
Running question 3/10...
  Running model: Llama-3.1-8B
  Running model: Llama-3.3-70B
  Running model: Llama-4-Scout
Running question 4/10...
  Running model: Llama-3.1-8B
  Running model: Llama-3.3-70B
  Running model: Llama-4-Scout
Running question 5/10...
  Running model: Llama-3.1-8B
  Running model: Llama-3.3-70B
  Running model: Llama-4-Scout
Running question 6/10...
  Running model: Llama-3.1-8B
  Running model: Llama-3.3-70B
  Running model: Llama-4-Scout
Running question 7/10...
  Running model: Llama-3.1-8B
  Running model: Llama-3.3-70B
  Running model: Llama-4-Scout
Running question 8/10...
  Running model: Llama-3.1-8B
  Running model: Llama-3.3-70B
  Running model: Llama-4-Scout
Running question 9/10...
  Running model: Llama-3.1-8B
  Running

In [15]:
# Print RAG answers
for _, row in df.iterrows():
    print(f"\n{'=' * 70}")
    print(f"Q{row['Q#']}: {row['Question']}")
    print(f"{'=' * 70}")

    for model_name in MODELS.keys():
        print(f"\n[{model_name}]")
        print(row[model_name])





Q1: What was the Supreme Court's ruling in Tinker v. Des Moines?

[Llama-3.1-8B]
The Supreme Court held that students do not "shed their constitutional rights to freedom of speech or expression at the schoolhouse gate."

[Llama-3.3-70B]
I cannot find this in the documents.

[Llama-4-Scout]
I cannot find this in the documents.

Q2: What legal standard did the court establish in Brandenburg v. Ohio?

[Llama-3.1-8B]
The court established that the government may not punish the advocacy of the use of force or lawless action, except where such advocacy is directed to inciting or producing imminent lawless action and is likely to incite or produce such action.

[Llama-3.3-70B]
I cannot find this in the documents.

[Llama-4-Scout]
The court established that "the mere advocacy of abstract ideas, or the formation and peaceful membership in groups espousing illegal action" is not punishable, but "incitement to imminent lawless action" is. Specifically, Chief Justice Hughes wrote: 

"The right of

In [16]:
# Scoring prompt
def score_answer(q_num, question, answer, context, ground_truth):
    scoring_prompt = f"""
You are evaluating an AI answer for a legal RAG project.

Question number:
{q_num}

Question:
{question}

Retrieved context:
{context}

Ground truth answer:
{ground_truth}

Model answer:
{answer}

Score the model answer using this format only:

Hallucination: 0 or 1
Accuracy: 0 or 1
Reasoning: 1 to 5

Rules:
- Hallucination = 1 if the answer includes fake citations, fake facts, unsupported claims, or information not found in the retrieved context.
- Hallucination = 0 if the answer stays grounded in the retrieved context.
- Accuracy = 1 if the answer is mostly correct compared with the ground truth.
- Accuracy = 0 if it is wrong, incomplete, or misleading.
- Reasoning should be from 1 to 5.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": scoring_prompt}
        ],
        max_tokens=150,
        temperature=0
    )

    return response.choices[0].message.content.strip()


In [17]:
# Parse scoring result
def parse_scores(raw_text):
    hallucination = 1
    accuracy = 0
    reasoning = 1

    h_match = re.search(r"Hallucination:\s*([01])", raw_text, re.IGNORECASE)
    a_match = re.search(r"Accuracy:\s*([01])", raw_text, re.IGNORECASE)
    r_match = re.search(r"Reasoning:\s*([1-5])", raw_text, re.IGNORECASE)

    if h_match:
        hallucination = int(h_match.group(1))

    if a_match:
        accuracy = int(a_match.group(1))

    if r_match:
        reasoning = int(r_match.group(1))

    return hallucination, accuracy, reasoning


In [18]:
# Score all model answers
results_df = pd.read_csv("lexrag_results.csv")
score_rows = []

for _, row in results_df.iterrows():
    q_num = int(row["Q#"])
    question = row["Question"]
    gt_answer = ground_truth_answers[q_num]

    retrieved = retriever.invoke(question)
    context = "\n\n".join([c.page_content for c in retrieved])

    score_row = {
        "Q#": q_num,
        "Question": question
    }

    print(f"Scoring Q{q_num}...")

    for model_name in MODELS.keys():
        answer = str(row[model_name])

        raw_score = score_answer(
            q_num=q_num,
            question=question,
            answer=answer,
            context=context,
            ground_truth=gt_answer
        )

        h, a, r = parse_scores(raw_score)

        score_row[f"{model_name}_hallucination"] = h
        score_row[f"{model_name}_accuracy"] = a
        score_row[f"{model_name}_reasoning"] = r

        time.sleep(1)

    score_rows.append(score_row)

scores_df = pd.DataFrame(score_rows)
scores_df.to_csv("lexrag_scores.csv", index=False)

print("Scores saved as lexrag_scores.csv")

Scoring Q1...
Scoring Q2...
Scoring Q3...
Scoring Q4...
Scoring Q5...
Scoring Q6...
Scoring Q7...
Scoring Q8...
Scoring Q9...
Scoring Q10...
Scores saved as lexrag_scores.csv


In [19]:
# Final summary table
scores_df = pd.read_csv("lexrag_scores.csv")

print("=" * 65)
print(f"{'LEXRAG — AUTOMATED SCORING SUMMARY':^65}")
print("=" * 65)
print(f"{'Model':<18} {'Hallucination':>14} {'Accuracy':>12} {'Reasoning':>12}")
print("-" * 65)

for model_name in MODELS.keys():
    hall_rate = scores_df[f"{model_name}_hallucination"].mean() * 100
    acc_rate = scores_df[f"{model_name}_accuracy"].mean() * 100
    avg_reason = scores_df[f"{model_name}_reasoning"].mean()

    print(
        f"{model_name:<18} "
        f"{hall_rate:>13.0f}% "
        f"{acc_rate:>11.0f}% "
        f"{avg_reason:>11.1f}/5"
    )

print("=" * 65)

best_accuracy = max(
    MODELS.keys(),
    key=lambda m: scores_df[f"{m}_accuracy"].mean()
)

lowest_hallucination = min(
    MODELS.keys(),
    key=lambda m: scores_df[f"{m}_hallucination"].mean()
)

best_reasoning = max(
    MODELS.keys(),
    key=lambda m: scores_df[f"{m}_reasoning"].mean()
)

print(f"\nBest accuracy     : {best_accuracy}")
print(f"Lowest hallucination: {lowest_hallucination}")
print(f"Best reasoning    : {best_reasoning}")

               LEXRAG — AUTOMATED SCORING SUMMARY                
Model               Hallucination     Accuracy    Reasoning
-----------------------------------------------------------------
Llama-3.1-8B                  10%          40%         2.8/5
Llama-3.3-70B                  0%          10%         1.9/5
Llama-4-Scout                  0%          20%         2.3/5

Best accuracy     : Llama-3.1-8B
Lowest hallucination: Llama-3.3-70B
Best reasoning    : Llama-3.1-8B


In [20]:
# Per-question breakdown
print("\n\nPER QUESTION BREAKDOWN:")
print("-" * 65)

for _, row in scores_df.iterrows():
    print(f"\nQ{int(row['Q#'])}: {row['Question'][:60]}...")

    for model_name in MODELS.keys():
        h = int(row[f"{model_name}_hallucination"])
        a = int(row[f"{model_name}_accuracy"])
        r = int(row[f"{model_name}_reasoning"])

        hall_label = "HALLUCINATED" if h == 1 else "grounded"

        print(
            f"   {model_name:<18} | "
            f"{hall_label:<13} | "
            f"accuracy:{a} | "
            f"reasoning:{r}/5"
        )




PER QUESTION BREAKDOWN:
-----------------------------------------------------------------

Q1: What was the Supreme Court's ruling in Tinker v. Des Moines?...
   Llama-3.1-8B       | grounded      | accuracy:1 | reasoning:4/5
   Llama-3.3-70B      | grounded      | accuracy:0 | reasoning:5/5
   Llama-4-Scout      | grounded      | accuracy:0 | reasoning:5/5

Q2: What legal standard did the court establish in Brandenburg v...
   Llama-3.1-8B       | grounded      | accuracy:1 | reasoning:5/5
   Llama-3.3-70B      | grounded      | accuracy:0 | reasoning:1/5
   Llama-4-Scout      | grounded      | accuracy:1 | reasoning:4/5

Q3: What is the 'actual malice' standard from New York Times v. ...
   Llama-3.1-8B       | HALLUCINATED  | accuracy:0 | reasoning:1/5
   Llama-3.3-70B      | grounded      | accuracy:0 | reasoning:1/5
   Llama-4-Scout      | grounded      | accuracy:0 | reasoning:1/5

Q4: Who wrote the majority opinion in Snyder v. Phelps and what ...
   Llama-3.1-8B       | groun

In [21]:
# Download CSV files
files.download("lexrag_results.csv")
files.download("lexrag_scores.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>